In [32]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("3")\
        .master("yarn")\
        .config("spark.executor.memory", "4g") \
        .config("'spark.executor.cores", "4")\
        .config("'spark.driver.cores", "4")\
        .config("spark.ui.port", "4051")\
        .getOrCreate()

#проверка

events = spark.read.json(path = "/user/master/data/events/date=2022-05-25")

events.show(10)




2025-03-25 12:47:06,244 WARN util.Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
2025-03-25 12:47:12,747 WARN util.Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
2025-03-25 12:47:12,750 WARN cluster.YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!
2025-03-25 12:47:17,129 WARN cluster.YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 1 for reason Container from a bad node: container_1729228596238_26237_01_000002 on host: rc1a-dataproc-d-5w4oxa2s5b8foehs.mdb.yandexcloud.net. Exit status: 137. Diagnostics: [2025-03-25 12:47:15.043]Container killed on request. Exit code is 137
[2025-03-25 12:47:15.043]Container exited with a non-zero exit code 137. 
[2025-03-25 12:47:15.043]Killed by external signal
.


+--------------------+------------+
|               event|  event_type|
+--------------------+------------+
|[,, 2022-05-25 17...|subscription|
|[,, 2022-05-25 15...|subscription|
|[,, 2022-05-25 18...|subscription|
|[,, 2022-05-25 00...|subscription|
|[,, 2022-05-25 03...|subscription|
|[,, 2022-05-25 04...|subscription|
|[,, 2022-05-25 15...|subscription|
|[,, 2022-05-25 01...|subscription|
|[,, 2022-05-25 00...|subscription|
|[,, 2022-05-25 07...|subscription|
+--------------------+------------+


In [1]:


from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("3")\
        .master("yarn")\
        .config("spark.executor.memory", "4g") \
        .config("'spark.executor.cores", "4")\
        .config("'spark.driver.cores", "4")\
        .config("spark.ui.port", "4051")\
        .getOrCreate()

#проверка

events = spark.read.json(path = "/user/master/data/events/date=2022-05-25")

events.show(10)

/opt/spark/conf/spark-env.sh: line 31: hadoop: command not found


26/06/05 12:46:10 WARN Utils: Your hostname, fv4l76n2jdl2n7frsa5i resolves to a loopback address: 127.0.1.1; using 10.130.0.9 instead (on interface eth0)
26/06/05 12:46:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/05 12:46:12 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/06/05 12:46:22 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/06/05 12:46:22 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


+--------------------+------------+
|               event|  event_type|
+--------------------+------------+
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
+--------------------+------------+
only showing top 10 rows



In [2]:
events = spark.read.json("/user/master/data/events")
events.write \
        .partitionBy("date", "event_type") \
        .mode("overwrite") \
        .parquet("/user/s28509656/data/events")

ЗАДАНИЕ 5.2.4

In [ ]:
import sys


import pyspark.sql.functions as F
import datetime




def find_candidates(messages, verified_tags, suggested_cutoff):
        all_tags = messages\
        .where("event.message_channel_to is not null")\
        .selectExpr(["event.message_from as user", "explode(event.tags) as tag"])\
        .groupBy("tag")\
        .agg(F.countDistinct("user").alias("suggested_count"))\
        .where(F.col("suggested_count") >= suggested_cutoff)

        return all_tags.join(verified_tags, "tag", "left_anti")
    
    


date = "2022-05-31"
days_count = 5
suggested_cutoff = 300
base_input_path = "/user/vasilyevol/data/events"
verified_tags_path = "/user/master/data/snapshots/tags_verified/actual"
base_output_path = "/user/vasilyevol/5.2.4/analytics/verified_tags_candidates_d5"



dt = datetime.datetime.strptime(date, '%Y-%m-%d')
paths = [f"{base_input_path}/date={(dt-datetime.timedelta(days=x)).strftime('%Y-%m-%d')}/event_type=message" for x in range(int(days_count))]
messages = spark.read.parquet(*paths)

verified_tags = spark.read.parquet(verified_tags_path)

candidates = find_candidates(messages, verified_tags, suggested_cutoff)

candidates.write.parquet(f"{base_output_path}/date={date}")


In [ ]:
!hadoop fs -ls /user/vasilyevol/data/analytics/candidates_d7_pyspark


5.3.1

In [5]:
import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window


def input_event_paths(date, depth):
    dt = datetime.datetime.strptime(date, '%Y-%m-%d')
    return [f"/user/s28509656/data/events/date={(dt - datetime.timedelta(days=x)).strftime('%Y-%m-%d')}" for x in
            range(depth)]


def tag_tops(date, depth, spark):
    message_paths = input_event_paths(date, depth)
    message_paths = input_event_paths(date, depth)
    result = spark.read\
    .option("basePath", "/user/s28509656/data/events")\
    .parquet(*message_paths)\
    .where("event_type = 'message'")\
    .where("event.message_channel_to is not null")\
    .select(F.col("event.message_id").alias("message_id"),
            F.col("event.message_from").alias("user_id"),
            F.explode(F.col("event.tags")).alias("tag"))\
    .groupBy("user_id", "tag")\
    .agg(F.count("*").alias("tag_count"))\
    .withColumn("rank", F.row_number().over(Window.partitionBy("user_id")\
    .orderBy(F.desc("tag_count"), F.desc("tag"))))\
    .where("rank <= 3")\
    .groupBy("user_id")\
    .pivot("rank", [1, 2, 3])\
    .agg(F.first("tag"))\
    .withColumnRenamed("1", "tag_top_1")\
    .withColumnRenamed("2", "tag_top_2")\
    .withColumnRenamed("3", "tag_top_3")
    result.write.mode("overwrite").parquet(f"/user/s28509656/data/tmp/tag_tops_{date[5:7]}_{date[8:10]}_{depth}")

    return None








In [6]:
depths=[5,5,1]
dates=['2022-06-04', '2022-05-04', '2022-05-04']

# dates=[ '2022-04-04', '2022-04-04']

s = zip(dates,depths)
for date,depth in s:
    tag_tops(date, depth, spark)

5.3.2

In [7]:
import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window


def input_event_paths(date, depth):
    dt = datetime.datetime.strptime(date, '%Y-%m-%d')
    return [f"/user/s28509656/data/events/date={(dt - datetime.timedelta(days=x)).strftime('%Y-%m-%d')}" for x in
            range(depth)]


def reaction_tag_tops(date, depth, spark):
    reaction_paths = input_event_paths(date, depth)
    reactions = spark.read\
    .option("basePath", "/user/s28509656/data/events")\
    .parquet(*reaction_paths)\
    .where("event_type='reaction'")

    all_message_tags = spark.read.parquet("/user/s28509656/data/events")\
    .where("event_type='message' and event.message_channel_to is not null")\
    .select(F.col("event.message_id").alias("message_id"),
            F.col("event.message_from").alias("user_id"),
            F.explode(F.col("event.tags")).alias("tag")
    )

    reaction_tags = reactions\
    .select(F.col("event.reaction_from").alias("user_id"), 
            F.col("event.message_id").alias("message_id"), 
            F.col("event.reaction_type").alias("reaction_type")
    ).join(all_message_tags.select("message_id", "tag"), "message_id")

    reaction_tops = reaction_tags\
    .groupBy("user_id", "tag", "reaction_type")\
    .agg(F.count("*").alias("tag_count"))\
    .withColumn("rank", F.row_number().over(Window.partitionBy("user_id", "reaction_type")\
    .orderBy(F.desc("tag_count"), F.desc("tag"))))\
    .where("rank <= 3")\
    .groupBy("user_id", "reaction_type")\
    .pivot("rank", [1, 2, 3])\
    .agg(F.first("tag"))\
    .cache()

    like_tops = reaction_tops\
    .where("reaction_type = 'like'")\
    .drop("reaction_type")\
    .withColumnRenamed("1", "like_tag_top_1")\
    .withColumnRenamed("2", "like_tag_top_2")\
    .withColumnRenamed("3", "like_tag_top_3")

    dislike_tops = reaction_tops\
    .where("reaction_type = 'dislike'")\
    .drop("reaction_type")\
    .withColumnRenamed("1", "dislike_tag_top_1")\
    .withColumnRenamed("2", "dislike_tag_top_2")\
    .withColumnRenamed("3", "dislike_tag_top_3")

    result = like_tops\
    .join(dislike_tops, "user_id", "full_outer")
    print("here 1 ")
    result.write.mode("overwrite").parquet(f"/user/s28509656/data/tmp/reaction_tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    return None




In [8]:
depths=[5,5,1]
dates=['2022-05-04', '2022-04-04', '2022-04-04']
s = zip(dates,depths)
for date,depth in s:
    reaction_tag_tops(date, depth, spark)

here 1 


here 1 


here 1 


5.3.3

In [10]:
depths=[5,5,1]
dates=['2022-04-04', '2022-04-05', '2022-04-04']
# dates=[ '2022-04-04', '2022-04-04']

s = zip(dates,depths)
for date,depth in s:
    tag_tops(date, depth, spark)

In [13]:
import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window
depths=[5,5,1]
dates=['2022-05-04', '2022-04-04', '2022-04-04']
s = zip(dates,depths)

def calculate_user_interests(date, depth, spark):
    print(f"/user/s28509656/data/tmp/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops = spark.read.parquet(f"/user/s28509656/data/tmp/tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    reaction_tops = spark.read.parquet(f"/user/s28509656/data/tmp/reaction_tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops.join(reaction_tops, "user_id", "full_outer").write.mode("overwrite").\
    parquet(f"/user/s28509656/data/tmp/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    
    
for date,depth in s:
    calculate_user_interests(date, depth, spark)



/user/s28509656/data/tmp/user_interests_05_04_5


/user/s28509656/data/tmp/user_interests_04_04_5


/user/s28509656/data/tmp/user_interests_04_04_1


In [ ]:
https://gist.github.com/goel52/547c0701d800ffde506c5ba7e8f36995

In [11]:
def calculate_user_interests(date, depth, spark):
    print(f"/user/vasilyevol/data/tmp/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops = spark.read.parquet(f"/tmp/data/tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    reaction_tops = spark.read.parquet(f"/tmp/data/reaction_tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops.join(reaction_tops, "user_id", "full_outer").write.mode("overwrite").\
    parquet(f"/tmp/data/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    
    
for date,depth in s:
    calculate_user_interests(date, depth, spark)

In [12]:
depths=[5,5,1]
dates=['2022-04-04', '2022-04-05', '2022-04-04']
for date,depth in s:
    calculate_user_interests(date, depth, spark)

In [ ]:
import sys
import datetime
import logging
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

def input_event_paths(date, depth, events_base_path):
    # Преобразуем дату в формат datetime
    end_date = datetime.datetime.strptime(date, "%Y-%m-%d")
    # Список путей к событиям
    event_paths = []
    
    # Создание списка путей для каждого дня в заданном диапазоне глубины (depth)
    for i in range(depth):
        day = (end_date - datetime.timedelta(days=i)).strftime("%Y-%m-%d")
        event_paths.append(f"{events_base_path}/date={day}")
    
    return event_paths

def calculate_user_interests(date, depth, events_base_path, output_base_path):
    logging.info("Initializing Spark session...")
    # Инициализация Spark-сессии
    spark = SparkSession.builder \
        .appName(f"User Interests Calculation for {date} with depth {depth}") \
        .master("yarn")\
        .config("spark.executor.memory", "4g")\
        .config("spark.executor.cores", "4")\
        .config("spark.driver.cores", "4")\
        .config("spark.ui.port", "4051")\
        .getOrCreate()
    
    # Получение путей к событиям
    event_paths = input_event_paths(date, depth, events_base_path)
    
    
    
    # Чтение данных событий
    logging.info("Reading event data...")
    events = spark.read.option("basePath", events_base_path).parquet(*event_paths)
    logging.info(f"Total events read: {events.count()}")

    # Выборка постов пользователя
    logging.info("Selecting user posts...")
    user_posts = events \
        .where("event_type = 'message' and event.message_channel_to is not null") \
        .select(F.col("event.message_id").alias("message_id"),
                F.col("event.message_from").alias("user_id"),
                F.explode(F.col("event.tags")).alias("tag"))
    logging.info(f"Total user posts selected: {user_posts.count()}")

    # Расчет топ-3 тегов по постам пользователя
    logging.info("Calculating top-3 tags for user posts...")
    user_post_tops = user_posts \
        .groupBy("user_id", "tag") \
        .agg(F.count("*").alias("tag_count")) \
        .withColumn("rank", F.row_number().over(Window.partitionBy("user_id")
                                                .orderBy(F.desc("tag_count"), F.asc("tag")))) \
        .where("rank <= 3") \
        .groupBy("user_id") \
        .pivot("rank", [1, 2, 3]) \
        .agg(F.first("tag")) \
        .withColumnRenamed("1", "tag_top_1") \
        .withColumnRenamed("2", "tag_top_2") \
        .withColumnRenamed("3", "tag_top_3")
    logging.info("User post tops calculated.")
    user_post_tops.show(5)

    # Выборка реакций пользователя
    logging.info("Selecting user reactions...")
    reactions = events.where("event_type='reaction'")
    logging.info(f"Total reactions selected: {reactions.count()}")

    # Теги сообщений для реакций
    logging.info("Selecting message tags for reactions...")
    message_tags = events \
        .where("event_type='message' and event.message_channel_to is not null") \
        .select(F.col("event.message_id").alias("message_id"),
                F.explode(F.col("event.tags")).alias("tag"))
    logging.info(f"Total message tags selected: {message_tags.count()}")

    # Соединение тегов реакций с тегами сообщений
    logging.info("Joining reaction tags with message tags...")
    reaction_tags = reactions \
        .select(F.col("event.reaction_from").alias("user_id"),
                F.col("event.message_id").alias("message_id"),
                F.col("event.reaction_type").alias("reaction_type")) \
        .join(message_tags, "message_id")
    logging.info(f"Total reaction tags after join: {reaction_tags.count()}")

    # Расчет топ-3 тегов по реакциям (лайки и дизлайки вместе)
    logging.info("Calculating top-3 tags for reactions...")
    reaction_tops = reaction_tags \
        .groupBy("user_id", "tag", "reaction_type") \
        .agg(F.count("*").alias("tag_count")) \
        .withColumn("rank", F.row_number().over(Window.partitionBy("user_id", "reaction_type")
                                                .orderBy(F.desc("tag_count"), F.asc("tag")))) \
        .where("rank <= 3") \
        .groupBy("user_id", "reaction_type") \
        .pivot("rank", [1, 2, 3]) \
        .agg(F.first("tag")) \
        .withColumnRenamed("1", "tag_top_1") \
        .withColumnRenamed("2", "tag_top_2") \
        .withColumnRenamed("3", "tag_top_3")
    logging.info("Reaction tops calculated.")
    reaction_tops.show(5)

    # Расчет топ-3 тегов по лайкам
    logging.info("Calculating top-3 tags for likes...")
    like_tops = reaction_tops \
        .where("reaction_type = 'like'") \
        .drop("reaction_type") \
        .withColumnRenamed("tag_top_1", "like_tag_top_1") \
        .withColumnRenamed("tag_top_2", "like_tag_top_2") \
        .withColumnRenamed("tag_top_3", "like_tag_top_3")
    like_tops.show(5)

    # Расчет топ-3 тегов по дизлайкам
    logging.info("Calculating top-3 tags for dislikes...")
    dislike_tops = reaction_tops \
        .where("reaction_type = 'dislike'") \
        .drop("reaction_type") \
        .withColumnRenamed("tag_top_1", "dislike_tag_top_1") \
        .withColumnRenamed("tag_top_2", "dislike_tag_top_2") \
        .withColumnRenamed("tag_top_3", "dislike_tag_top_3")
    dislike_tops.show(5)
    
    # Очистка кэша Spark
    spark.catalog.clearCache()

    # Объединение всех результатов в один датафрейм
    logging.info("Joining all results into one dataframe...")
    result = user_post_tops \
        .join(like_tops, "user_id", "full_outer") \
        .join(dislike_tops, "user_id", "full_outer")
    logging.info("Result dataframe created.")
    result.show(5)

    # Остановка Spark-сессии
    logging.info("Stopping Spark session.")
    result.write.parquet(f'{output_dir}/user_interests_{date.replace("-", "_")}_{days_count}')
    

    # Возврат результирующего датафрейма
    return result

if __name__ == "__main__":
    # Аргументы
    date = "2022-05-04"
    days_count = 5
    events_base_path = "/user/vasilyevol/data/events"
    output_base_path = "/user/vasilyevol/analytics/user_interests_d5"

    # Выполнение расчета и сохранение результата
    result_df = calculate_user_interests(date, days_count, events_base_path, output_base_path)
    logging.info("h1")
#     result_df.write.mode("overwrite").parquet(f'{output_dir}/user_interests_{date.replace("-", "_")}_{days_count}')
    



In [ ]:
!hadoop fs -ls /user/vasilyevol/analytics/user_interests_d5

In [ ]:
df = spark.read.parquet("/user/vasilyevol/analytics/user_interests_d5")

In [ ]:
df.show(truncate=False)

In [ ]:
import sys

from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import datetime

def input_event_paths(base_path, date, depth):
    dt = datetime.datetime.strptime(date, '%Y-%m-%d')
    return [f"{base_path}/date={(dt-datetime.timedelta(days=x)).strftime('%Y-%m-%d')}" for x in range(int(depth))]

def main():
    date = sys.argv[1]
    days_count = sys.argv[2]
    events_base_path = sys.argv[3]
    interests_base_path = sys.argv[4]
    verified_tags_path = sys.argv[5]
    output_base_path = sys.argv[6]

    conf = SparkConf().setAppName(f"ConnectionInterestsJob-{date}-d{days_count}")
    sc = SparkContext(conf=conf)
    sql = SQLContext(sc)

    messages = sql.read\
    .option("basePath", events_base_path)\
    .parquet(*input_event_paths(events_base_path, date, days_count))\
    .where("event_type='message'")

    direct_messages = messages.where("event.message_to is not null")
    posts = messages.where("event.message_channel_to is not null")

    interests = sql.read.parquet(f"{interests_base_path}/date={date}")
    subscriptions = sql.read.parquet(events_base_path)\
    .where(f"event_type = 'subscription' and date <= '{date}'")

    verified_tags = sql.read.parquet(verified_tags_path)

    contacts = get_contacts(direct_messages)
    contact_interests = get_contact_interests(contacts, interests).cache()
    subs_interests = get_subs_interests(posts, subscriptions, verified_tags).cache()

    result = join_result(contact_interests, subs_interests)
    result.write.mode("overwrite").parquet(f"{output_base_path}/date={date}")


def get_contacts(direct_messages):
    return direct_messages\
    .select(F.col("event.message_from").alias("from"),
        F.col("event.message_to").alias("to"),
        F.explode(F.array(F.col("event.message_from"), F.col("event.message_to"))).alias("user_id"))\
    .withColumn("contact_id", F.when(F.col("user_id") == F.col("from"), F.col("to")).otherwise(F.col("from")))\
    .select("user_id", "contact_id")\
    .distinct()

def get_contact_interests(contacts, interests):
    return contacts\
    .withColumnRenamed("user_id", "u")\
    .join(interests, F.col("contact_id") == F.col("user_id"))\
    .transform(lambda df: add_tag_usage_count(df, "u"))\
    .transform(lambda df: add_tag_rank(df, "u"))\
    .groupBy("u")\
    .agg(*[top_direct_tag(c) for c in result_columns()])\
    .withColumnRenamed("u", "user_id")

def get_subs_interests(posts, subscriptions, verified_tags):
    post_tags = posts\
    .select(F.col("event.message_channel_to").alias("channel_id"),
    F.col("event.tags").alias("tags"))

    verified_sub_tags = post_tags\
    .join(subscriptions,
    (F.col("event.subscription_channel") == F.col("channel_id")))\
    .select(F.col("event.user").alias("user_id"), F.explode(F.col("tags")).alias("tag"))\
    .join(verified_tags, "tag", "left_semi")

    return verified_sub_tags\
    .groupBy("user_id", "tag")\
    .agg(F.countDistinct("*").alias("tag_count"))\
    .withColumn("rank", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("tag_count"), F.desc("tag"))))\
    .where("rank <= 3")\
    .groupBy("user_id")\
    .pivot("rank", [1, 2, 3])\
    .agg(F.first("tag"))\
    .withColumnRenamed("1", "sub_verified_tag_top_1")\
    .withColumnRenamed("2", "sub_verified_tag_top_2")\
    .withColumnRenamed("3", "sub_verified_tag_top_3")

def join_result(contact_interests, subs_interests):
    return contact_interests\
        .join(subs_interests, "user_id", "full_outer")

def tag_columns(reaction):
    return [f"{reaction}_tag_top_{X}" for X in range(1, 4)]

def result_columns():
    return tag_columns("like") + tag_columns("dislike") 

def add_tag_usage_count(df, key):
    res = df
    cols = result_columns()
    for c in cols:
        res = res.withColumn(c+'_count', F.count(c).over(Window.partitionBy(key, c)))
    return res    

def add_tag_rank(df, key):
    res = df
    for c in result_columns():
        res = res.withColumn(c+"_rank", F.row_number().over(Window.partitionBy(key).orderBy(F.desc(c), F.desc(c+"_count"))))
    return res

def top_direct_tag(column):
    return F.first(F.when(F.col(column+"_rank") == 1, F.col(column)), True).alias('direct_'+column)


if __name__ == "__main__":
    main()

In [ ]:
/usr/lib/spark/bin/spark-submit --master yarn --deploy-mode cluster connection_interests.py 2022-05-25 7 /user/prod/data/events /user/prod/data/analytics/user_interests_d7 /user/master/data/snapshots/tags_verified/actual /user/prod/data/analytics/connection_interests_d7

In [ ]:
!hadoop fs -ls /user/master/data/geo/events

In [ ]:
df = spark.read.parquet("/user/master/data/geo/events/date=2022-01-23")

In [ ]:
df.selectExpr('even

In [ ]:
df.printSchema()

In [ ]:
!hadoop fs -ls /tmp/data

In [ ]:

import datetime
import pyspark.sql.functions as F
def input_paths(date, depth):
    dt = datetime.datetime.strptime(date, '%Y-%m-%d')
    return [f"/user/vasilyevol/data/events/date={(dt-datetime.timedelta(days=x)).strftime('%Y-%m-%d')}\
             event_type=message"
            for x in range(depth)]

In [ ]:
paths = input_paths('2022-05-31', 7)
messages = spark.read.parquet(*paths)
all_tags = messages.where("event.message_channel_to is not null")

In [ ]:
all_tags.count()

In [ ]:
all_tags.show()